# BeeHave Honey Yield Model — Training & Export

This notebook contains only the reproducible training pipeline used for the deployed honey-yield model.

### Pipeline
1. Load the BeeHave raw 144×40 daily sequences.
2. Extract environmental features using the same 3-hour windows as the original BeeHave preprocessing.
3. Select the final 29 environmental features.
4. Train a `RandomForestRegressor`.
5. Validate the saved pipeline with a sample prediction.
6. Export the deployable `.pkl` model.

Exploratory MLP experiments, feature-search experiments, plots, duplicate training code, and intermediate model variants are intentionally excluded from this repository version.


## 1. Imports and final feature configuration

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


TOP_29_FEATURES = [
    "WeekSin",
    "T_in_16",
    "CO2_13",
    "T_in_grad_07",
    "HumidOut_13",
    "T_in_grad_19",
    "HumidOut_16",
    "T_in_10",
    "Pr_01",
    "WeekCos",
    "CO2_16",
    "T_in_grad_16",
    "T_out_feel_16",
    "T_out_feel_13",
    "T_out_feel_10",
    "CO2_07",
    "Pr_22",
    "T_diff_01",
    "T_in_22",
    "T_out_22",
    "Wind_16",
    "rain_04",
    "Wind_10",
    "Wind_04",
    "T_out_feel_01",
    "T_diff_22",
    "Wind_01",
    "Wind_19",
    "T_out_19"
]

ENV_FEATURES = [
    "WeekSin",
    "WeekCos",

    # Internal temperature
    "T_in_01", "T_in_04", "T_in_07", "T_in_10",
    "T_in_13", "T_in_16", "T_in_19", "T_in_22",

    # Internal temperature gradient
    "T_in_grad_01", "T_in_grad_04", "T_in_grad_07", "T_in_grad_10",
    "T_in_grad_13", "T_in_grad_16", "T_in_grad_19", "T_in_grad_22",

    # External temperature
    "T_out_01", "T_out_04", "T_out_07", "T_out_10",
    "T_out_13", "T_out_16", "T_out_19", "T_out_22",

    # Feels-like temperature
    "T_out_feel_01", "T_out_feel_04", "T_out_feel_07", "T_out_feel_10",
    "T_out_feel_13", "T_out_feel_16", "T_out_feel_19", "T_out_feel_22",

    # Temperature difference
    "T_diff_01", "T_diff_04", "T_diff_07", "T_diff_10",
    "T_diff_13", "T_diff_16", "T_diff_19", "T_diff_22",

    # Internal humidity
    "HumidIn_01", "HumidIn_04", "HumidIn_07", "HumidIn_10",
    "Humidin_13", "Humidin_16", "Humidin_19", "Humidin_22",

    # External humidity
    "HumidOut_01", "HumidOut_04", "HumidOut_07", "HumidOut_10",
    "HumidOut_13", "HumidOut_16", "HumidOut_19", "HumidOut_22",

    # Wind
    "Wind_01", "Wind_04", "Wind_07", "Wind_10",
    "Wind_13", "Wind_16", "Wind_19", "Wind_22",

    # Rain
    "rain_01", "rain_04", "rain_07", "rain_10",
    "rain_13", "rain_16", "rain_19", "rain_22",

    # CO2
    "CO2_01", "CO2_04", "CO2_07", "CO2_10",
    "CO2_13", "CO2_16", "CO2_19", "CO2_22",

    # Pressure
    "Pr_01", "Pr_04", "Pr_07", "Pr_10",
    "Pr_13", "Pr_16", "Pr_19", "Pr_22"
]

## 2. Load BeeHave data

In [ ]:
DATA_PATH = "/kaggle/input/beehave/AllFeatures_shuffled_144.npy"

AllFeatures = np.load(DATA_PATH, allow_pickle=True)
AllFeaturesN = AllFeatures.transpose(0, 2, 1)

print("Raw data shape:", AllFeaturesN.shape)


## 3. Feature engineering

In [ ]:
class BeeHaveEnvironmentalFeatureEngineering(
    BaseEstimator,
    TransformerMixin
):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = np.asarray(X, dtype=object)

        # -------------------------------------------------
        # Expected raw BeeHave shape
        # (samples, 144, 40)
        # -------------------------------------------------

        if X.ndim != 3:
            raise ValueError(
                f"Expected 3D input (samples, 144, 40), "
                f"got {X.shape}"
            )

        if X.shape[1] != 144:
            raise ValueError(
                f"Expected 144 time points, got {X.shape[1]}"
            )

        if X.shape[2] != 40:
            raise ValueError(
                f"Expected 40 raw channels, got {X.shape[2]}"
            )

        # -------------------------------------------------
        # EXACT SAME 3-HOUR WINDOWS AS Script.py
        # -------------------------------------------------

        windows = [
            (0, 18),       # 01
            (18, 36),      # 04
            (36, 54),      # 07
            (54, 72),      # 10
            (72, 90),      # 13
            (90, 108),     # 16
            (108, 126),   # 19
            (126, 144)    # 22
        ]

        time_labels = [
            "01",
            "04",
            "07",
            "10",
            "13",
            "16",
            "19",
            "22"
        ]

        # -------------------------------------------------
        # EXACT ENVIRONMENTAL CHANNEL MAPPING
        #
        # Script.py processes indices 4-34.
        # The first 11 are environmental.
        # Audio starts after these.
        # -------------------------------------------------

        channel_map = {
            4:  "T_in",
            5:  "T_in_grad",
            6:  "T_out",
            7:  "T_out_feel",
            8:  "T_diff",
            9:  "HumidIn",
            10: "HumidOut",
            11: "Wind",
            12: "rain",
            13: "CO2",
            14: "Pr"
        }

        rows = []

        for i in range(X.shape[0]):

            row = {}

            # -------------------------------------------------
            # Week / Season
            # Same as Script.py
            # -------------------------------------------------

            row["WeekSin"] = float(X[i, 0, 2])
            row["WeekCos"] = float(X[i, 0, 3])

            # -------------------------------------------------
            # Environmental features
            # 3-hour mean
            # -------------------------------------------------

            for channel_idx, feature_name in channel_map.items():

                for (start, end), time_label in zip(
                    windows,
                    time_labels
                ):

                    values = np.asarray(
                        X[i, start:end, channel_idx],
                        dtype=float
                    )

                    row[
                        f"{feature_name}_{time_label}"
                    ] = float(values.mean())

            rows.append(row)

        df = pd.DataFrame(rows)

        # -------------------------------------------------
        # Match BeeHave's original capitalization
        # -------------------------------------------------

        df.rename(
            columns={
                "HumidIn_13": "Humidin_13",
                "HumidIn_16": "Humidin_16",
                "HumidIn_19": "Humidin_19",
                "HumidIn_22": "Humidin_22"
            },
            inplace=True
        )

        return df

## 4. Top-29 feature selector

In [ ]:
class BeeHaveTop29Selector(
    BaseEstimator,
    TransformerMixin
):

    def __init__(self, features=None):
        self.features = features

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        missing = [
            f for f in self.features
            if f not in X.columns
        ]

        if missing:
            raise ValueError(
                f"Missing selected features: {missing}"
            )

        return X[self.features]

## 5. Build deployable pipeline

In [ ]:
beehave_pipeline = Pipeline([
    (
        "feature_engineering",
        BeeHaveEnvironmentalFeatureEngineering()
    ),
    (
        "top29_selection",
        BeeHaveTop29Selector(features=TOP_29_FEATURES)
    ),
    (
        "random_forest",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=29,
            oob_score=True,
            max_samples=800,
            random_state=13,
            n_jobs=-1
        )
    )
])

print("Pipeline created successfully.")


## 6. Prepare target

In [ ]:
frames = AllFeaturesN[:, 143, 35].astype(float)
weight_change_per_frame = AllFeaturesN[:, 143, 36].astype(float)

# Daily hive weight change
y = weight_change_per_frame * frames

print("Target shape:", y.shape)
print("First 5 target values:", y[:5])


## 7. Train

In [ ]:
beehave_pipeline.fit(AllFeaturesN, y)

rf = beehave_pipeline.named_steps["random_forest"]

print("Training completed.")
print("OOB R²:", rf.oob_score_)


## 8. Validate the trained pipeline

In [ ]:
predictions = beehave_pipeline.predict(AllFeaturesN[:10])

mae = mean_absolute_error(y[:10], predictions)
rmse = np.sqrt(mean_squared_error(y[:10], predictions))

print("Sample predictions:", predictions)
print("Sample MAE:", mae)
print("Sample RMSE:", rmse)


## 9. Export deployable model

In [ ]:
MODEL_PATH = "BeeHave_Environmental_Pipeline.pkl"

joblib.dump(beehave_pipeline, MODEL_PATH)

print(f"Saved deployable model: {MODEL_PATH}")


## Repository note

The exported file `BeeHave_Environmental_Pipeline.pkl` must be kept in sync with the ML service.

The custom transformer classes in this notebook are part of the serialized pipeline, so the corresponding Python class definitions must remain available to the service when the `.pkl` is loaded.
